In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.ndimage import gaussian_filter
from scipy.stats import zscore

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
samples = np.loadtxt(f'{projdir}/txt/allc_table/donor_leiden_fibro.txt', dtype=str)
mask = np.array(['Fibro' in x for x in samples])
samples = samples[mask]
samples.shape

In [ ]:
def process_bed(s):
    pmds_mc = pd.read_csv(f'{projdir}/bed/pmds/donor_leiden_fibro/{s}.start_pmds_CGN-Both.full.bed.gz', sep='\t', header=None)
    pmds_mc.index = [f'{x}_{y}' for x,y in zip(pmds_mc[0], pmds_mc[1])]
    return pmds_mc

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_bed, samples), total=len(samples)))

In [ ]:
results[0].head()

In [ ]:
mc_results = [x[3] / x[4] for x in results]
pmds_df = pd.concat(mc_results, axis=1)
pmds_df.columns = samples

for s in samples:
    pmds_df[s].fillna(pmds_df[s].mean(), inplace=True)
    
pmds_df = pmds_df[~pmds_df.index.duplicated()]
pmds_df = pmds_df.loc[:,sorted(pmds_df.columns)]
pmds_df.shape

In [ ]:
pmds_df.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    mask = [d in x for x in pmds_df.columns]
    tmp_df = pmds_df.loc[:,mask]
    sns.heatmap(tmp_df, vmin=0.1, vmax=0.7, cmap='coolwarm', ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=6)
    ax.set_yticklabels([]);

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
pmds_df.to_csv(f'{outdir}/mcg_leiden_fibro_pmd_mcg.csv', sep='\t')

In [ ]:
!date